# 1.0 Importando pacotes

In [1]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
import os

# 2.0 Definindo caminhos

In [2]:
# Caminho da pasta de inputs
inputs_path = Path('./inputs').resolve()

# Caminho da pasta de outputs
outputs_path = Path('./outputs').resolve()


# Caminhos dos subsets
subset_co_path = inputs_path / 'subset_co.parquet'
subset_no2_path = inputs_path / 'subset_no2.parquet'
subset_o3_path = inputs_path / 'subset_o3.parquet'
subset_pm_path = inputs_path / 'subset_pm.parquet'
subset_so2_path = inputs_path / 'subset_so2.parquet'

# 3.0 Carregando subsets

In [3]:
subset_co = gpd.read_parquet(subset_co_path)
subset_no2 = gpd.read_parquet(subset_no2_path)
subset_o3 = gpd.read_parquet(subset_o3_path)
subset_pm = gpd.read_parquet(subset_pm_path)
subset_so2 = gpd.read_parquet(subset_so2_path)


## 3.1 Montando tabelas de referência

In [4]:
# Lendo os arquivos para cada tabela
# https://www.gov.br/mma/pt-br/assuntos/meio-ambiente-urbano-recursos-hidricos-qualidade-ambiental/qualidade-do-ar/guia-tecnico-para-o-monitoramento-e-avaliacao-da-qualidade-do-ar.pdf

ref_table_co = pd.read_csv(inputs_path / 'ref_table_so2eco.csv')
ref_table_no2 = pd.read_csv(inputs_path / 'ref_table_no2.csv')
ref_table_o3 = pd.read_csv(inputs_path / 'ref_table_o3.csv')
ref_table_pm = pd.read_csv(inputs_path / 'ref_table_pm.csv')
ref_table_so2 = pd.read_csv(inputs_path / 'ref_table_so2eco.csv')

# Nota da EPA acerca da ocorrência de valores intermediários de ADT
'''
Distance from the edge of the nearest traffic lane. The distance for 
intermediate traffic counts should be interpolated from the table values based
on the actual traffic count.

Distância da borda da via mais próxima. A distância para contagens de veículos (ADT)
intermediárias deve ser interpolada a partir dos valores das tabelas baseados na
contagem de veículos observada.

# https://www.ecfr.gov/current/title-40/chapter-I/subchapter-C/part-58/appendix-Appendix%20E%20to%20Part%2058
'''

# Dicionário de faixas de ADT (real = valor * 1000)
pollutants_adt_dict = {'co':[1, 10, 20, 30, 40, 50, 60, np.inf],
                       'so2':[1, 10, 20, 30, 40, 50, 60, np.inf],
                       'no2':[1, 10, 15, 20, 40, 70, 110, np.inf],
                       'pm': [1, 15, 20, 30, 40, 50, 60, 70, 80, np.inf],
                       'o3':[10, 15, 20, 40, 70, 110, np.inf]}

# Definindo coluna de adt como índice
interpolated_co = ref_table_co.set_index('avg_adt').squeeze()
interpolated_no2 = ref_table_no2.set_index('avg_adt').squeeze()
interpolated_o3 = ref_table_o3.set_index('avg_adt').squeeze()
interpolated_pm = ref_table_pm.set_index('avg_adt').squeeze()
interpolated_so2 = ref_table_so2.set_index('avg_adt').squeeze()


## 3.2 Interpolação dos limites de distância para cada classe de representatividade espacial

In [5]:
## Interpolando os limites das classes de representatividade ------------------------
def interp_limites_rep(subset: gpd.GeoDataFrame,
                       interpolated: pd.DataFrame,
                       poluente: str) -> pd.DataFrame:
  
    # Dicionário de faixas de ADT (value * 1000)
    pollutants_adt_dict = {'co':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'so2':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'no2':[1, 10, 15, 20, 40, 70, 110, np.inf],
                           'pm': [1, 15, 20, 30, 40, 50, 60, 70, 80, np.inf],
                           'o3':[10, 15, 20, 40, 70, 110, np.inf]}
    
    # Iterando sobre os valores de adt da tabela de referencia de cada poluente
    for idx, adt_band in enumerate(pollutants_adt_dict[poluente][:-1]):
        col_name = f'average_daily_vehicle_count_{adt_band}k'
        
        if col_name not in subset.columns:
            print(f"[WARNING] '{col_name}' nonexistant in subset_{poluente}")
            continue
        
        # Iterando sobre os valores de adt para cada via do subset do poluente
        for adt_value in subset[col_name]:
            interpolated.loc[adt_value] = np.nan
                
    # Organizar pelo índice de modo ascendente
    interpolated.sort_index(inplace=True)
                
    # Interpolando os valores NaN #FIXME
    cols = [col for col in interpolated.columns if (('min' in col) | ('max' in col))]
    for col in cols:
        vals = set(interpolated[col].dropna().unique())
        if len(vals) == 1 and interpolated[col].isna().any():
            interpolated[col] = interpolated[col].dropna().unique()[0]
        else:
            interpolated[col] = pd.to_numeric(interpolated[col],
                                              errors='coerce')
            
            interpolated[col] = interpolated[col].interpolate(method='index',
                                                              limit_area='inside')
            
            last_valid = interpolated[col].last_valid_index()
            if last_valid is not None:
                interpolated.loc[last_valid:, col] = interpolated.loc[last_valid, col]
            interpolated[col] = interpolated[col].fillna(np.inf)

                
    # Resetando index
    interpolated.reset_index(inplace=True)

    return interpolated

In [6]:
# Aplicando função
interpolated_co = interp_limites_rep(subset_co,
                                     interpolated_co,
                                     'co')
interpolated_no2 = interp_limites_rep(subset_no2,
                                     interpolated_no2,
                                     'no2')
interpolated_o3 = interp_limites_rep(subset_o3,
                                     interpolated_o3,
                                     'o3')
interpolated_pm = interp_limites_rep(subset_pm,
                                     interpolated_pm,
                                     'pm')
interpolated_so2 = interp_limites_rep(subset_so2,
                                     interpolated_so2,
                                     'so2')

# 4.0 Salvando Outputs

In [7]:
interpolated_co.to_parquet(outputs_path / 'interpolated_co.parquet')
interpolated_no2.to_parquet(outputs_path / 'interpolated_no2.parquet')
interpolated_o3.to_parquet(outputs_path / 'interpolated_o3.parquet')
interpolated_pm.to_parquet(outputs_path / 'interpolated_pm.parquet')
interpolated_so2.to_parquet(outputs_path / 'interpolated_so2.parquet')


In [8]:
ref_table_co

,avg_adt,micro_min,micro_max,bairro_min,bairro_max
0,1000,2,10,10,inf
1,10000,2,10,10,inf
2,15000,2,10,25,inf
3,20000,2,10,35,inf
4,30000,2,10,80,inf
5,40000,2,10,115,inf
6,50000,2,10,135,inf
7,60000,2,10,150,inf


In [9]:
interpolated_co

,avg_adt,micro_min,micro_max,bairro_min,bairro_max
0,1000.000000,2.0,10.0,10.0,inf
1,1001.167856,2.0,10.0,10.0,inf
2,1023.157497,2.0,10.0,10.0,inf
3,1027.004026,2.0,10.0,10.0,inf
4,1032.606861,2.0,10.0,10.0,inf
...,...,...,...,...,...
1010,118363.632769,2.0,10.0,150.0,inf
1011,119087.250319,2.0,10.0,150.0,inf
1012,136695.970485,2.0,10.0,150.0,inf
1013,138996.156595,2.0,10.0,150.0,inf
